# Đánh giá pipeline từng cell (Google Colab)

Chạy **từng cell**: `Shift+Enter`. **Không** bấm Runtime → Run all khi đang đo thời gian.

So với pipeline cũ của bạn:

| Pipeline cũ (PDF OCR) | Pipeline này |
| --- | --- |
| Bước 1: pandoc ~0.4s | Bước 1: OOXML extract (`.docx`) ~0.2s |
| Bước 3: OCR 4 ảnh ~49s / ~57s mỗi trang | Bước 3: UniMERNet chỉ crop công thức; Unlimited-OCR từng trang nếu PDF |
| Bước 4: ghép+fix ~0.4s | Bước 4: markdown → Azota / gắn LaTeX |

Đầu ra Azota: `markup.txt` + `sidecar/` + `manifest.json`.


## 0. Cấu hình — chạy cell này trước


In [ ]:
# Runtime → Change runtime type → GPU T4 (A100 nếu bật Unlimited-OCR)
REPO_URL = "https://github.com/phuchoang2603/refurbished-marketplace.git"
REPO_BRANCH = "cursor/docx-to-azota-pipeline-4d56"

RUN_UNIMERNET = True
UNIMERNET_SIZE = "tiny"          # tiny | small | base
MAX_UNIMERNET_IMAGES = 4         # None = hết; 4 ≈ log "OCR 4 ảnh" của bạn

RUN_UNLIMITED_OCR = False        # True trên A100; T4 dễ OOM
OCR_GUNDAM = True
MAX_OCR_PAGES = 1                # None = hết 5 trang (~286s như log Kaggle)

OUT_DIR = "/content/azota_out"
print("OK config")


In [ ]:
!nvidia-smi -L || echo "CPU only — Bước 3 sẽ chậm"


## 0b. Clone converter


In [ ]:
import sys, shutil
from pathlib import Path

if Path("/content/docx-to-azota").exists():
    shutil.rmtree("/content/docx-to-azota")
!git clone -b {REPO_BRANCH} --depth 1 {REPO_URL} /content/refurbished-marketplace
src = Path("/content/refurbished-marketplace/tools/docx-to-azota")
shutil.copytree(src, "/content/docx-to-azota")
sys.path.insert(0, "/content/docx-to-azota")
print("cloned", src)


In [ ]:
from convert import convert_docx, apply_unimernet_latex, write_ocr_sidecar
from eval_timer import StepTimer
from markdown_to_azota import markdown_to_azota
from vision import (
    rasterize_formula_image,
    load_unimernet,
    unimernet_one,
    load_unlimited_ocr,
    unlimited_ocr_one,
    strip_unlimited_ocr_det,
    pdf_to_images,
)
print("import OK")


## 0c. Cài package — tách 3 cell để đo thời gian install


In [ ]:
# CPU: luôn cần
!pip -q install pillow pymupdf
print("CPU deps OK")


In [ ]:
# UniMERNet — không dùng [full] (tokenizers source build fail trên Colab)
if RUN_UNIMERNET:
    !apt-get -qq install -y imagemagick libmagickwand-dev >/dev/null
    from install_colab import allow_wmf_in_imagemagick, install_unimernet_colab
    allow_wmf_in_imagemagick()
    install_unimernet_colab()
    print("UniMERNet deps OK")
else:
    print("SKIP UniMERNet install")


In [ ]:
# Unlimited-OCR ~3B (bỏ qua nếu False)
if RUN_UNLIMITED_OCR:
    !pip -q install -U transformers==4.57.1 einops addict easydict
    !apt-get -qq install -y libreoffice >/dev/null
    print("Unlimited-OCR deps OK")
else:
    print("SKIP Unlimited-OCR install")


## 1. Upload đề

Chọn **một** file: `.docx` (OOXML, nhanh) hoặc `.pdf` (OCR từng trang, giống log Kaggle ~57s/trang).


In [ ]:
from google.colab import files

uploaded = files.upload()
if uploaded:
    INPUT_PATH = "/content/" + next(iter(uploaded))
else:
    INPUT_PATH = "/content/docx-to-azota/samples/de-vat-li-lan-3.docx"
INPUT_PATH = str(Path(INPUT_PATH))
SUFFIX = Path(INPUT_PATH).suffix.lower()
print("INPUT:", INPUT_PATH, "kind:", SUFFIX)


In [ ]:
from pathlib import Path
timer = StepTimer()
Path(OUT_DIR).mkdir(parents=True, exist_ok=True)
JOBS = []
PAGES = []
MANIFEST = None
PREDS = {}
OCR_PAGES_MD = []
UM_MODEL = None
OCR_MODEL = None
print("timer sẵn sàng — các Bước 1–4 sẽ cộng vào đây")


## Bước 1 — extract (thay pandoc)

- `.docx` → OOXML walker (giữ `[!m:$mathml_N$]`, `*D.`, bảng).
- `.pdf` → raster PNG (tương đương “PDF to PNG 0.37s” trong log của bạn).


In [ ]:
import json
from pathlib import Path

PAGES = []
MANIFEST = None
MARKUP_PATH = Path(OUT_DIR) / "markup.txt"

if SUFFIX == ".docx":
    with timer.step("Bước 1", "OOXML extract"):
        MANIFEST = convert_docx(INPUT_PATH, OUT_DIR)
    print(json.dumps(MANIFEST["counts"], ensure_ascii=False, indent=2))
elif SUFFIX == ".pdf":
    with timer.step("Bước 1", "PDF → PNG"):
        PAGES = pdf_to_images(INPUT_PATH, dpi=200)
    print(f"Số trang/ảnh: {len(PAGES)}")
    for i, p in enumerate(PAGES, 1):
        print(f"  page_{i:04d}: {p}")
else:
    raise SystemExit(f"Cần .docx hoặc .pdf, nhận được {SUFFIX}")


### Đánh giá Bước 1 — xem 40 dòng markup (chỉ `.docx`)


In [ ]:
if MARKUP_PATH.exists():
    lines = MARKUP_PATH.read_text(encoding="utf-8").splitlines()
    print(f"Số dòng: {len(lines)}")
    print("--- 40 dòng đầu ---")
    print("\n".join(lines[:40]))
else:
    print("PDF path: chưa có markup.txt (sẽ có sau Bước 3+4)")


In [ ]:
# Liệt kê sidecar (docx) hoặc file trang (pdf)
from pathlib import Path
sc = Path(OUT_DIR) / "sidecar"
if sc.exists():
    files = sorted(sc.iterdir())
    print(f"sidecar: {len(files)} files")
    for p in files[:25]:
        print(f"  {p.name:30s} {p.stat().st_size:8d} B")
    if len(files) > 25:
        print(f"  ... +{len(files)-25}")
else:
    print("chưa có sidecar/")


## Bước 2 — raster WMF MathType → PNG

Chỉ chạy với `.docx`. Đây là khâu chuẩn bị ảnh công thức cho UniMERNet (không OCR cả trang).


In [ ]:
JOBS = []
if SUFFIX == ".docx" and MANIFEST:
    png_dir = Path(OUT_DIR) / "sidecar_png"
    png_dir.mkdir(exist_ok=True)
    with timer.step("Bước 2", "raster WMF"):
        for asset in MANIFEST["assets"]:
            if asset["kind"] != "mathtype":
                continue
            src = Path(OUT_DIR) / asset["sidecar"]
            dest = png_dir / f"{asset['id']}.png"
            got = rasterize_formula_image(src, dest)
            if got:
                JOBS.append((asset["id"], got))
    print(f"{len(JOBS)} ảnh công thức")
    if MAX_UNIMERNET_IMAGES:
        JOBS = JOBS[:MAX_UNIMERNET_IMAGES]
        print(f"cắt còn {len(JOBS)} ảnh để đánh giá (MAX_UNIMERNET_IMAGES)")
else:
    print("SKIP Bước 2 (không phải docx / không có MathType)")


In [ ]:
# Xem 1 ảnh công thức đã raster
from IPython.display import Image, display
if JOBS:
    aid, path = JOBS[0]
    print(aid, path)
    display(Image(filename=str(path)))
else:
    print("không có ảnh để preview")


## Bước 3a — UniMERNet từng ảnh (`.docx`)

Tách **load model** và **infer từng ảnh** để bạn thấy thời gian load vs thời gian/ảnh.


In [ ]:
UM_MODEL = None
if RUN_UNIMERNET and JOBS:
    !git clone --depth 1 https://github.com/opendatalab/UniMERNet.git /content/UniMERNet || true
    !mkdir -p /content/UniMERNet/models
    ckpt = {"tiny": "unimernet_tiny", "small": "unimernet_small", "base": "unimernet_base"}[UNIMERNET_SIZE]
    if not Path(f"/content/UniMERNet/models/{ckpt}").exists():
        !git lfs install
        !git clone --depth 1 https://huggingface.co/wanderkid/{ckpt} /content/UniMERNet/models/{ckpt}
    with timer.step("Bước 3a-load", "load UniMERNet"):
        UM_MODEL = load_unimernet(cfg_path="/content/UniMERNet/configs/demo.yaml")
    print("model device:", UM_MODEL[2])
else:
    print("SKIP load UniMERNet")


In [ ]:
PREDS = {}
if UM_MODEL and JOBS:
    model, vis_processor, device = UM_MODEL
    with timer.step("Bước 3", f"UniMERNet {len(JOBS)} ảnh"):
        for i, (aid, path) in enumerate(JOBS, 1):
            t = StepTimer()
            with t.step(f"  ảnh {i}/{len(JOBS)}", aid):
                PREDS[aid] = unimernet_one(model, vis_processor, device, path)
            print(f"    {aid} → {PREDS[aid][:100]}")
else:
    print("SKIP infer UniMERNet")


In [ ]:
if PREDS and MANIFEST:
    apply_unimernet_latex(MANIFEST, PREDS, Path(OUT_DIR))
    print("đã ghi sidecar/*.tex + manifest.latex")
    print(list(PREDS.items())[:3])
else:
    print("không có LaTeX để gắn")


## Bước 3b — Unlimited-OCR từng trang (`.pdf` hoặc QA hình vẽ)

Giống log Kaggle: in thời gian **từng trang**. Mặc định `MAX_OCR_PAGES = 1` để đánh giá nhanh; đặt `None` để chạy hết (~57s × N trang).


In [ ]:
if RUN_UNLIMITED_OCR and SUFFIX == ".docx" and not PAGES:
    !soffice --headless --convert-to pdf --outdir /content "{INPUT_PATH}"
    pdfs = list(Path("/content").glob("*.pdf"))
    if pdfs:
        PAGES = pdf_to_images(str(pdfs[0]), dpi=200)
        print("docx→pdf→", len(PAGES), "trang")
    else:
        print("LibreOffice không tạo PDF")
elif not PAGES:
    print("không có trang để OCR (bật RUN_UNLIMITED_OCR hoặc upload PDF)")


In [ ]:
OCR_MODEL = None
if RUN_UNLIMITED_OCR and PAGES:
    with timer.step("Bước 3b-load", "load Unlimited-OCR"):
        OCR_MODEL = load_unlimited_ocr()
    print("Unlimited-OCR loaded")
else:
    print("SKIP load Unlimited-OCR")


In [ ]:
OCR_PAGES_MD = []
pages = list(PAGES)
if MAX_OCR_PAGES:
    pages = pages[:MAX_OCR_PAGES]
if OCR_MODEL and pages:
    model, tokenizer = OCR_MODEL
    print(f"Số trang/ảnh: {len(pages)}")
    with timer.step("Bước 3", f"OCR {len(pages)} ảnh"):
        for i, page in enumerate(pages, 1):
            t = StepTimer()
            out_p = f"{OUT_DIR}/ocr_raw/page_{i:04d}"
            with t.step(f"Trang {i}", Path(page).name):
                raw = unlimited_ocr_one(model, tokenizer, page, out_p, gundam=OCR_GUNDAM)
            cleaned = strip_unlimited_ocr_det(raw)
            OCR_PAGES_MD.append(cleaned)
            print(cleaned[:400])
            print("---")
else:
    print("SKIP OCR trang")


## Bước 4 — ghép + fix → Azota

- `.docx`: markup đã là Azota; bước này chỉ xác nhận.
- `.pdf` / OCR: Markdown+LaTeX (như screenshot trái) → `[!b:$…$]`, `[img:$img_N$]`, `[* c1 | c2 *]`.


In [ ]:
if OCR_PAGES_MD:
    md_all = "\n\n".join(OCR_PAGES_MD)
    with timer.step("Bước 4", "ghép+fix markdown→Azota"):
        text, assets = markdown_to_azota(
            md_all,
            sidecar_dir=Path(OUT_DIR) / "sidecar",
            media_root=Path("/"),
        )
        MARKUP_PATH.write_text(text, encoding="utf-8")
        extra = {"source": Path(INPUT_PATH).name, "ocr_assets": assets, "pages": len(OCR_PAGES_MD)}
        (Path(OUT_DIR) / "manifest_ocr.json").write_text(
            json.dumps(extra, ensure_ascii=False, indent=2), encoding="utf-8"
        )
    print(text[:1500])
elif SUFFIX == ".docx":
    with timer.step("Bước 4", "ghép+fix (đã có trong extract)"):
        pass
    print("docx: không cần ghép OCR — markup.txt đã Azota")
else:
    print("không có dữ liệu để ghép")


## Tổng thời gian (format giống log của bạn)


In [ ]:
timer.print_summary()


## QA Azota + tải zip


In [ ]:
import re, json
from pathlib import Path
from google.colab import files

out = Path(OUT_DIR)
text = (out / "markup.txt").read_text(encoding="utf-8") if (out / "markup.txt").exists() else ""
print("dòng:", len(text.splitlines()))
print("mathml", text.count("[!m:$mathml_"), "mathtype", text.count("[!m:$mathtype_"), "img", text.count("[img:$img_"))
print("MCQ *", re.findall(r"\*[A-D]\.", text)[:20])
print("TF *", re.findall(r"\*[a-d]\)", text)[:20])
print("Đáp án ngắn", re.findall(r"→ Đáp án:.*", text))
print("--- 25 dòng ---")
print("\n".join(text.splitlines()[:25]))


In [ ]:
from google.colab import files
!cd /content && zip -qr azota_out.zip azota_out
files.download("/content/azota_out.zip")
